# 예제 franka_ex04: FR3 끝단 자세 지령 — Pose Goal (self-contained)

Franka FR3(7-DOF) 의 끝단(`fr3_hand_tcp`) 위치(`x, y, z`) 와 방향(`roll, pitch, yaw`) 을
지정해 이동하는 예제. 6-DOF 용 `ex04_pose_goal.ipynb` 를 FR3 워크스페이스 크기에 맞춰 옮겨온 버전이다.

**6-DOF 예제와 다른 점**
- 끝단 링크: `fr3_hand_tcp` (그리퍼 손가락 사이 TCP 프레임)
- 워크스페이스: FR3 의 reach 가 약 855mm 이므로 목표를 더 멀리(>30cm) / 더 높게(z~0.5m) 잡는다
- 7-DOF redundancy 덕에 같은 pose 도 IK 해가 여러 개 → 플래너가 보통 더 자연스러운 경로를 찾는다
- `home`(올-제로) 자세가 SRDF 에 없다 → 기준 자세는 `ready` 만 사용
- Gazebo Sim 위에서 도는 환경이라 `use_sim_time=True` 가 필요하다

**학습 내용**
- `PositionConstraint`, `OrientationConstraint` 메시지 구성
- IK(역기구학) 솔버의 역할 — pose 는 입력, 조인트 값은 출력
- Euler ↔ Quaternion 변환
- RViz2 `MarkerArray` 로 목표 자세를 **미리** 시각화

**워크플로**

각 목표마다 두 단계로 나뉜다:

1. **미리보기 단계** — `preview_pose_target(...)` 호출. RViz2 에 마커만 찍히고 로봇은 움직이지 않는다.
2. **실행 단계** — `go_to_pose_goal(...)` 호출. MoveIt 이 IK + 모션 플래닝을 수행하고 실제로 로봇이 이동한다.

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는 방식이다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/pose_goal_markers` 로 설정**한다.
이 토픽으로 미리보기 마커가 발행된다.
Fixed Frame 은 `fr3_link0` 로 두면 된다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab franka_ex04_pose_goal.ipynb
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 로봇 상수 정의

이 값들은 `franka_description/robots/fr3/fr3.srdf.xacro` 에서 생성되는 SRDF 와 일치한다.

In [1]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/pose_goal_markers'

## 2. ROS 2 초기화와 노드 생성

`MoveGroup` 액션 클라이언트, `MarkerArray` 퍼블리셔, `joint_states` 구독자를 같은 노드에 붙인다.
Gazebo Sim 과 시간을 맞추기 위해 `use_sim_time=True` 를 준다.

### 2-1. import (이 섹션에서 처음 쓰이는 것들)

In [2]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup
from visualization_msgs.msg import MarkerArray

### 2-2. `rclpy` 초기화

한 프로세스에서 한 번만 init 가능하므로, 두 번째 실행은 무시되도록 `try/except` 로 감싼다.

In [3]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

### 2-3. 노드, 액션 클라이언트, 마커 퍼블리셔, `joint_states` 구독자

In [4]:
node = Node(
    'franka_ex04_pose_goal_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex04 노트북 노드 생성 완료 ===')

[INFO] [1778393517.020520834] [franka_ex04_pose_goal_demo]: === franka_ex04 노트북 노드 생성 완료 ===


True

## 3. 액션 서버와 `/joint_states` 준비 대기

이 셀에서 `time` 모듈이 처음 쓰이므로 함께 import 한다.

In [5]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

[INFO] [1778393517.924623460] [franka_ex04_pose_goal_demo]: action server + /joint_states 준비됨


## 4. SRDF 에서 `ready` 자세 읽어오기

SRDF 파라미터를 받아 group_state 를 dict 로 파싱한다.
여기서 `AsyncParameterClient` 와 `xml.etree.ElementTree` 가 처음 쓰이므로 같이 import 한다.
FR3 SRDF 에는 `home` 이 없고 `ready` / `extended` 만 있다.

In [6]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

[INFO] [1778393519.181774264] [franka_ex04_pose_goal_demo]: ready: {'fr3_joint1': 0.0, 'fr3_joint2': -0.7853981633974483, 'fr3_joint3': 0.0, 'fr3_joint4': -2.356194490192345, 'fr3_joint5': 0.0, 'fr3_joint6': 1.5707963267948966, 'fr3_joint7': 0.7853981633974483}


True

## 5. 결론부터 — 사용자가 부르는 함수 3개

이 노트북에서 사용자가 직접 호출하는 함수는 세 가지뿐이다.

| 함수 | 결과 | 로봇 이동? |
|---|---|---|
| `preview_pose_target(target)` | RViz 에 마커만 발행 | ✗ 정지 |
| `check_pose_reachable(pose)` | IK + 모션 플래닝까지만 (도달 가능성 확인) | ✗ 정지 |
| `go_to_pose_goal(pose)` | IK + 플래닝 + 컨트롤러 송신 | ✓ 이동 |

뒤의 두 함수는 본질적으로 같은 액션을 보낸다 — 차이는 `PlanningOptions.plan_only` 한 비트뿐이다.

`go_to_pose_goal` 을 가장 바깥으로 두고 한 겹씩 벗겨보면:

| 깊이 | 함수 | 역할 |
|---|---|---|
| (1) 사용자 호출 | `go_to_pose_goal` | `Pose` 한 개 → IK + 플래닝 + 실행 |
| (2) 입력 타입 | `make_pose` (`euler_to_quaternion`) | x,y,z + roll,pitch,yaw → `Pose` 메시지 |
| (3) 액션 송수신 | `send_goal_and_wait` | **핵심: `move_client.send_goal_async(goal)`** + accept/result 대기 |
| (4) 데이터 타입 | `make_goal` / `make_plan_request` / `make_position_constraint` / `make_orientation_constraint` | `goal` 이 받아야 할 메시지 트리를 채워 만드는 작업 |

이외에 두 갈래가 더 있다:

- **동반 도구** (7장) — `preview_pose_target` (마커), `check_pose_reachable` (plan_only)
- **옆가지** (8장) — `ready` 자세는 SRDF 조인트값이라 Pose 가 아니라 `JointConstraint` 로 가야 하므로 `go_to_joint_goal` 도 있다 (ex03 와 동일 흐름)


## 6. `go_to_pose_goal` 한 겹씩 벗기기

`Pose` 한 개 → 액션 한 번 송신 → 로봇 이동. 이 한 줄짜리 흐름의 안쪽을 입력부터 데이터 타입 가장 안쪽까지 차례로 본다.

### 6-1. 입력 타입 — `Pose` 와 `make_pose` (Euler → Quaternion)

`go_to_pose_goal(pose)` 의 인자는 `geometry_msgs/Pose`. 위치는 `Point(x, y, z)`, 방향은 **쿼터니언**(`x, y, z, w`).

사람이 입력하기 편한 `roll/pitch/yaw` (rad) 을 쿼터니언으로 바꿔주는 헬퍼 `euler_to_quaternion`,
그리고 위치/방향을 한 번에 받아 `Pose` 로 묶어주는 `make_pose` 를 먼저 만든다.

(여기서 처음 쓰이는 `math`, `tf_transformations`, `Pose / Point / Quaternion` 을 함께 import.)

In [7]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

### 6-2. 가장 바깥 — `go_to_pose_goal`

`Pose` 한 개를 받아서 두 가지를 한다:

1. Pose → `PositionConstraint` + `OrientationConstraint` 두 개로 분해해 `Constraints` 한 덩어리로 묶는다
2. `Constraints` 를 `MotionPlanRequest` 에 넣고 `MoveGroup.Goal` 로 감싸 `send_goal_and_wait` 로 송신

리턴은 success 여부. IK 해가 없거나 충돌이 있으면 실패한다 — 코드값을 로깅한다.

(아래 def 는 `make_plan_request`, `make_position_constraint`, `make_orientation_constraint`, `make_goal`,
`send_goal_and_wait`, `MoveItErrorCodes` 를 호출 시점에 바인딩한다. 이들은 6-3 부터 차례로 정의된다.)

In [8]:
def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                     attempts: int = 5, plan_time: float = 10.0) -> bool:
    req = make_plan_request(vel, acc, attempts, plan_time)
    constraints = Constraints()
    constraints.position_constraints.append(make_position_constraint(pose))
    constraints.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(constraints)
    code_val = send_goal_and_wait(make_goal(req))
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

### 6-3. 한 단계 안 — `send_goal_and_wait` 와 **핵심** `send_goal_async`

ex03 와 동일한 함수다. 이 한 줄이 모든 일의 시작:

```python
send_future = move_client.send_goal_async(goal)   # ← 핵심
```

ROS 2 액션은 비동기라 future 를 두 번 기다린다 — accept future, result future.
리턴은 `MoveItErrorCodes` 정수 (성공 = `SUCCESS` = 1).

이번엔 `goal` 안의 `Constraints` 가 `JointConstraint` 가 아니라 `PositionConstraint` + `OrientationConstraint` 라는 점만 다르다.

In [9]:
from moveit_msgs.msg import MoveItErrorCodes

def send_goal_and_wait(goal: MoveGroup.Goal) -> int:
    send_future = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, send_future)
    handle = send_future.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED
    result_future = handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)
    return result_future.result().result.error_code.val

### 6-4. `goal` 의 데이터 타입 — Pose 버전

ex03 의 트리와 거의 같은데, 가장 안쪽이 다르다. **조인트값** 대신 **위치/방향 제약** 두 종류가 들어간다.

```
MoveGroup.Goal
├─ request : MotionPlanRequest
│  ├─ group_name = 'fr3_arm'
│  ├─ num_planning_attempts, allowed_planning_time
│  ├─ max_velocity_scaling_factor, max_acceleration_scaling_factor
│  └─ goal_constraints : [Constraints]
│     ├─ position_constraints    : [PositionConstraint]   ← 위치 (구로 둘러싼 영역)
│     └─ orientation_constraints : [OrientationConstraint] ← 방향 (쿼터니언 + 축별 tolerance)
└─ planning_options : PlanningOptions   ← plan_only 등 옵션
```

이 트리도 가장 안쪽부터 채워 올라간다.

#### 6-4-1. 가장 안쪽 — `PositionConstraint` + `OrientationConstraint`

- **`PositionConstraint`** — 끝단(`fr3_hand_tcp`) 이 들어가야 할 영역을 *구* 로 정의. `BoundingVolume` 안에 `SolidPrimitive(SPHERE)` 와 그 위치 Pose 를 담는다.
- **`OrientationConstraint`** — 끝단의 방향(쿼터니언) + x/y/z 축별 절대 tolerance.

이렇게 분해된 두 제약이 같은 `Constraints` 한 덩어리에 묶여 들어간다.

In [10]:
from moveit_msgs.msg import (
    Constraints, PositionConstraint, OrientationConstraint, BoundingVolume,
)
from geometry_msgs.msg import Vector3
from shape_msgs.msg import SolidPrimitive

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)

    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)

    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)

    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose: Pose, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    oc.orientation = pose.orientation
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

#### 6-4-2. 그 위 — `MotionPlanRequest` 에 담기

플랜 리퀘스트 본체. 그룹 이름, 시도 횟수, 허용 시간, 속도/가속 스케일을 채운다.
`goal_constraints` 추가는 `go_to_pose_goal` 안에서 하므로 여기선 비어 있는 채로 만들어둔다.
(8장의 조인트 목표도 같은 함수를 재사용한다.)

In [11]:
from moveit_msgs.msg import MotionPlanRequest

def make_plan_request(vel: float, acc: float,
                       attempts: int, plan_time: float) -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    return req

#### 6-4-3. 마지막 껍데기 — `MoveGroup.Goal` + `PlanningOptions`

액션 goal 메시지로 한 번 더 감싼다. `PlanningOptions` 는 액션 동작을 결정하는 옵션:

- `plan_only=False` : 컨트롤러로 trajectory 송신 (= 실제 이동)
- `plan_only=True` : 플래닝까지만, 송신 안 함 (← 7-2 의 `check_pose_reachable` 가 쓰는 모드)
- `replan=True, replan_attempts=3` : 도중에 실패하면 다시 시도

In [12]:
from moveit_msgs.msg import PlanningOptions

def make_goal(req: MotionPlanRequest) -> MoveGroup.Goal:
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=False, replan=True, replan_attempts=3)
    return goal

## 7. 동반 도구

`go_to_pose_goal` 옆에서 같이 쓰는 두 함수.

### 7-1. 미리보기 — `preview_pose_target` (RViz 마커만, 로봇 정지)

`/pose_goal_markers` 토픽으로 마커 3 개를 발행한다 — 화살표(arrow), 위치 강조용 구(sphere), 라벨(text).
RViz 에서 `MarkerArray` Display 를 추가하고 토픽을 `/pose_goal_markers` 로 설정해 두면 보인다.

색상 의미: 노란색 = 미리보기, 파란색 = 진행 중, 초록색 = 성공, 빨간색 = 실패.

여기서 처음 쓰이는 `ColorRGBA`, `Marker`, `Vector3` 가 함께 import 된다.

In [13]:
from std_msgs.msg import ColorRGBA
from visualization_msgs.msg import Marker
from geometry_msgs.msg import Vector3

COLOR_PENDING = ColorRGBA(r=1.0, g=1.0, b=0.0, a=0.8)   # 노란색
COLOR_ACTIVE  = ColorRGBA(r=0.2, g=0.5, b=1.0, a=0.9)   # 파란색
COLOR_SUCCESS = ColorRGBA(r=0.0, g=1.0, b=0.0, a=0.8)   # 초록색
COLOR_FAIL    = ColorRGBA(r=1.0, g=0.0, b=0.0, a=0.8)   # 빨간색
COLOR_TEXT    = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)   # 흰색

# 발행한 마커들의 누적 상태 (덮어쓰기 위해 보관)
_markers = MarkerArray()

def _build_markers(idx: int, target: dict, color: ColorRGBA) -> list:
    stamp = node.get_clock().now().to_msg()
    pose = make_pose(target['x'], target['y'], target['z'],
                     target['roll'], target['pitch'], target['yaw'])

    arrow = Marker()
    arrow.header.frame_id = REFERENCE_FRAME
    arrow.header.stamp = stamp
    arrow.ns = 'pose_goal_arrow'
    arrow.id = idx
    arrow.type = Marker.ARROW
    arrow.action = Marker.ADD
    arrow.pose = pose
    arrow.scale = Vector3(x=0.18, y=0.025, z=0.025)
    arrow.color = color

    sphere = Marker()
    sphere.header.frame_id = REFERENCE_FRAME
    sphere.header.stamp = stamp
    sphere.ns = 'pose_goal_sphere'
    sphere.id = idx
    sphere.type = Marker.SPHERE
    sphere.action = Marker.ADD
    sphere.pose.position = Point(x=target['x'], y=target['y'], z=target['z'])
    sphere.pose.orientation.w = 1.0
    sphere.scale = Vector3(x=0.06, y=0.06, z=0.06)
    sphere.color = ColorRGBA(r=color.r, g=color.g, b=color.b, a=0.6)

    text = Marker()
    text.header.frame_id = REFERENCE_FRAME
    text.header.stamp = stamp
    text.ns = 'pose_goal_text'
    text.id = idx
    text.type = Marker.TEXT_VIEW_FACING
    text.action = Marker.ADD
    text.pose.position = Point(x=target['x'], y=target['y'], z=target['z'] + 0.15)
    text.pose.orientation.w = 1.0
    text.scale.z = 0.07
    text.color = COLOR_TEXT
    text.text = target.get('label', f'target {idx}')
    return [arrow, sphere, text]

def publish_marker(idx: int, target: dict, color: ColorRGBA) -> None:
    """같은 (ns, id) 마커는 덮어쓰고, 누적 MarkerArray 를 한 번에 발행."""
    new = _build_markers(idx, target, color)
    keys = {(m.ns, m.id) for m in new}
    _markers.markers = [m for m in _markers.markers if (m.ns, m.id) not in keys]
    _markers.markers.extend(new)
    marker_pub.publish(_markers)

def preview_pose_target(target: dict, idx: int = 1) -> None:
    """목표 자세를 RViz2 에 노란색 미리보기 마커로 표시. 로봇은 움직이지 않는다.

    target dict: {'label': str, 'x': float, 'y': float, 'z': float,
                  'roll': rad, 'pitch': rad, 'yaw': rad}
    """
    publish_marker(idx, target, COLOR_PENDING)
    node.get_logger().info(
        f"[preview] idx={idx} {target.get('label', '')} "
        f"pos=({target['x']:.2f}, {target['y']:.2f}, {target['z']:.2f}) "
        f"rpy=({math.degrees(target['roll']):.0f}°, "
        f"{math.degrees(target['pitch']):.0f}°, "
        f"{math.degrees(target['yaw']):.0f}°)"
    )

### 7-2. 도달 가능성 검사 — `check_pose_reachable` (plan_only)

`go_to_pose_goal` 과 같은 액션을 보내되 `PlanningOptions.plan_only=True` 로 설정한다.
MoveIt 이 IK + 플래닝까지만 수행하고 컨트롤러로는 송신하지 않는다 → 로봇은 가만히 있는다.

리턴은 `(ok, error_code)`. 자주 보는 코드:

| 코드 | 이름 | 의미 |
|---|---|---|
| `1` | `SUCCESS` | 도달 가능 |
| `-1` | `PLANNING_FAILED` | 경로 못 찾음 |
| `-12` | `GOAL_IN_COLLISION` | 목표에서 충돌 |
| `-31` | `NO_IK_SOLUTION` | IK 해 없음 (작업 공간 밖) |

In [14]:
MOVEIT_ERROR_NAMES = {
      1: 'SUCCESS',
     -1: 'PLANNING_FAILED',
     -2: 'INVALID_MOTION_PLAN',
     -6: 'TIMED_OUT',
    -10: 'START_STATE_IN_COLLISION',
    -11: 'START_STATE_VIOLATES_PATH_CONSTRAINTS',
    -12: 'GOAL_IN_COLLISION',
    -13: 'GOAL_VIOLATES_PATH_CONSTRAINTS',
    -14: 'GOAL_CONSTRAINTS_VIOLATED',
    -15: 'INVALID_GROUP_NAME',
    -16: 'INVALID_GOAL_CONSTRAINTS',
    -17: 'INVALID_ROBOT_STATE',
    -27: 'GOAL_STATE_INVALID',
    -31: 'NO_IK_SOLUTION',
}

def check_pose_reachable(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                          attempts: int = 5, plan_time: float = 5.0):
    """Pose 목표에 IK + 모션 플래닝만 수행한다 (로봇은 움직이지 않음).

    Returns:
        (ok: bool, error_code: int)
    """
    req = make_plan_request(vel, acc, attempts, plan_time)
    constraints = Constraints()
    constraints.position_constraints.append(make_position_constraint(pose))
    constraints.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(constraints)

    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=True, replan=False, replan_attempts=0)

    code_val = send_goal_and_wait(goal)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    name = MOVEIT_ERROR_NAMES.get(code_val, f'code={code_val}')
    if ok:
        node.get_logger().info('  ✓ 도달 가능 — 경로 존재 확인')
    else:
        node.get_logger().warn(f'  ✗ 도달 불가 — {name}')
    return ok, code_val

## 8. 옆가지 — 조인트 목표 (`ready` 이동용)

`ready` 는 SRDF `group_state` 로 정의된 *조인트값* 이라, Pose 가 아니라 `JointConstraint` 로 보내야 한다.
구조는 ex03 와 동일 — `JointConstraint` 묶음 → `Constraints` → `MotionPlanRequest` → `MoveGroup.Goal`.
중간 단계 (`make_plan_request`, `make_goal`, `send_goal_and_wait`) 는 6장에서 만든 것을 그대로 재사용한다.

(여기서 처음 쓰이는 `JointConstraint` 만 추가로 import.)

In [15]:
from moveit_msgs.msg import JointConstraint

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    constraints = Constraints()
    for jname, val in joint_values.items():
        constraints.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return constraints

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 5.0) -> bool:
    req = make_plan_request(vel, acc, attempts, plan_time)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val = send_goal_and_wait(make_goal(req))
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

## 9. 시나리오 시작 — `ready` 자세로 초기화

Pose 목표를 보내기 전에 잘 보이는 자세(`ready`)로 옮긴다.

In [16]:
node.get_logger().info('--- ready 자세로 초기 이동 ---')
go_to_joint_goal(ready_target)
time.sleep(1.0)

[INFO] [1778393631.206532863] [franka_ex04_pose_goal_demo]: --- ready 자세로 초기 이동 ---


## 10. 첫 번째 목표 — 전방 (FR3 워크스페이스 안쪽)

끝단을 로봇 전방 (`x=0.50, y=0.0, z=0.50`) 으로 보내고, TCP 가 아래를 향하도록 `roll = π` 를 준다.
(FR3 reach 가 ~85cm 이므로 50cm 정도는 안전한 안쪽이다.)

### 10-1. 목표 정의 + 미리보기 (마커만 발행, 로봇은 움직이지 않음)

In [17]:
target1 = {
    'label': 'Forward',
    'x': 0.50, 'y': 0.00, 'z': 0.50,
    'roll': math.pi, 'pitch': 0.0, 'yaw': 0.0,
}
preview_pose_target(target1, idx=1)

[INFO] [1778393635.608186585] [franka_ex04_pose_goal_demo]: [preview] idx=1 Forward pos=(0.50, 0.00, 0.50) rpy=(180°, 0°, 0°)


### 10-2. 마커 위치 확인했으면 실제로 이동

In [18]:
publish_marker(1, target1, COLOR_ACTIVE)
ok = go_to_pose_goal(make_pose(
    target1['x'], target1['y'], target1['z'],
    target1['roll'], target1['pitch'], target1['yaw'],
))
publish_marker(1, target1, COLOR_SUCCESS if ok else COLOR_FAIL)
time.sleep(1.0)

## 11. 두 번째 목표 — 좌측

끝단을 로봇 좌측 (`y=+0.45`) 으로 보내고 yaw 를 90° 돌려 손목을 좌측을 향하게 한다.

### 11-1. 미리보기

In [19]:
target2 = {
    'label': 'Left',
    'x': 0.10, 'y': 0.45, 'z': 0.50,
    'roll': math.pi, 'pitch': 0.0, 'yaw': math.pi / 2,
}
preview_pose_target(target2, idx=2)

[INFO] [1778393641.567804054] [franka_ex04_pose_goal_demo]: [preview] idx=2 Left pos=(0.10, 0.45, 0.50) rpy=(180°, 0°, 90°)


### 11-2. 실제로 이동

In [20]:
publish_marker(2, target2, COLOR_ACTIVE)
ok = go_to_pose_goal(make_pose(
    target2['x'], target2['y'], target2['z'],
    target2['roll'], target2['pitch'], target2['yaw'],
))
publish_marker(2, target2, COLOR_SUCCESS if ok else COLOR_FAIL)
time.sleep(1.0)

## 12. 세 번째 목표 — 높이 올린 자세

끝단을 위로 올린 자세 (`z=0.75`). FR3 의 위쪽 reach 한계 근처라 IK 가 빡빡할 수 있다 — 도달 실패하면 `target3` 의 z 값을 `0.65~0.70` 으로 낮춰 다시 시도해 보자.

### 12-1. 미리보기

In [21]:
target3 = {
    'label': 'High',
    'x': 0.30, 'y': 0.00, 'z': 0.75,
    'roll': math.pi, 'pitch': 0.0, 'yaw': 0.0,
}
preview_pose_target(target3, idx=3)

[INFO] [1778393647.933613001] [franka_ex04_pose_goal_demo]: [preview] idx=3 High pos=(0.30, 0.00, 0.75) rpy=(180°, 0°, 0°)


### 12-2. 실제로 이동

In [22]:
publish_marker(3, target3, COLOR_ACTIVE)
ok = go_to_pose_goal(make_pose(
    target3['x'], target3['y'], target3['z'],
    target3['roll'], target3['pitch'], target3['yaw'],
))
publish_marker(3, target3, COLOR_SUCCESS if ok else COLOR_FAIL)
time.sleep(1.0)

## 13. 보너스 — 직접 목표를 입력해 보기

세 단계로 나뉜다:

1. `preview_pose_target(my_target)` — 마커만 표시 (로봇은 가만히)
2. `check_pose_reachable(my_pose)` — 도달 가능한지 미리 확인 (실행 X)
3. 가능하면 `go_to_pose_goal(my_pose)` — 실제 이동

FR3 작업 공간의 한계를 시험해 보자. reach 가 약 85cm 이므로 base 로부터 거리가 그보다 크면 IK 해가 없다.

### 13-1. 미리보기

In [23]:
my_target = {
    'label': 'MyTarget',
    'x': 0.40, 'y': -0.25, 'z': 0.45,
    'roll': math.pi, 'pitch': 0.0, 'yaw': -math.pi / 4,
}
preview_pose_target(my_target, idx=99)

[INFO] [1778393653.637579616] [franka_ex04_pose_goal_demo]: [preview] idx=99 MyTarget pos=(0.40, -0.25, 0.45) rpy=(180°, 0°, -45°)


### 13-2. 도달 가능성 검사 (실행 X)

위의 `my_target` 이 정말 도달 가능한지 미리 확인한다.
`reachable=True` 면 다음 셀로 가서 이동, `False` 면 위로 올라가 `my_target` 값을 수정하고 다시 시도한다.

In [24]:
my_pose = make_pose(
    my_target['x'], my_target['y'], my_target['z'],
    my_target['roll'], my_target['pitch'], my_target['yaw'],
)
reachable, code_val = check_pose_reachable(my_pose)
print(f'reachable = {reachable}, error_code = {code_val}')

reachable = True, error_code = 1


[INFO] [1778393654.810706662] [franka_ex04_pose_goal_demo]:   ✓ 도달 가능 — 경로 존재 확인


### 13-3. 도달 가능하면 실제로 이동

In [25]:
publish_marker(99, my_target, COLOR_ACTIVE)
ok = go_to_pose_goal(my_pose)
publish_marker(99, my_target, COLOR_SUCCESS if ok else COLOR_FAIL)

## 14. `ready` 로 복귀

In [26]:
node.get_logger().info('--- ready 로 복귀 ---')
go_to_joint_goal(ready_target)
node.get_logger().info('=== franka_ex04 완료! ===')

[INFO] [1778393661.572071244] [franka_ex04_pose_goal_demo]: --- ready 로 복귀 ---
[INFO] [1778393664.001303489] [franka_ex04_pose_goal_demo]: === franka_ex04 완료! ===


True

## 15. 정리

노트북을 닫기 전에 노드와 rclpy 를 안전하게 정리한다.

In [27]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass